In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, upper, trim, to_date, when

# Initialize Spark Session
spark = SparkSession.builder.appName("PharmaPipeline").getOrCreate()
# 1. Clean Demographics Table
silver_demog = spark.read.table("bronze_demographics") \
    .filter(col("primaryid").isNotNull()) \
    .dropDuplicates(["primaryid"]) \
    .withColumn("event_date", to_date(col("event_dt"), "yyyyMMdd")) \
    .withColumn("patient_age", col("age").cast("integer")) \
    .withColumn("country", upper(trim(col("reporter_country")))) \
    .select("primaryid", "caseid", "event_date", "patient_age", "age_cod", "gndr_cod", "country")

silver_demog.write.format("delta").mode("overwrite").saveAsTable("silver_demographics")

# 2. Clean Drug Table (Standardize names and expand role codes)
silver_drug = spark.read.table("bronze_drug") \
    .filter(col("primaryid").isNotNull()) \
    .withColumn("clean_drug_name", upper(trim(col("drugname")))) \
    .withColumn("drug_role", when(col("role_cod") == "PS", "Primary Suspect")
                            .when(col("role_cod") == "SS", "Secondary Suspect")
                            .otherwise("Other")) \
    .select("primaryid", "clean_drug_name", "drug_role", "route")

silver_drug.write.format("delta").mode("overwrite").saveAsTable("silver_drug")

# 3. Clean Reaction Table
silver_reac = spark.read.table("bronze_reaction") \
    .filter(col("primaryid").isNotNull()) \
    .withColumn("reaction_term", upper(trim(col("pt")))) \
    .select("primaryid", "reaction_term")

silver_reac.write.format("delta").mode("overwrite").saveAsTable("silver_reaction")

print("Silver Layer tables successfully transformed and saved!")

Silver Layer tables successfully transformed and saved!
